<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/05_instrucciones_opcional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 05 · (Opcional) Ajuste por instrucciones: de «modelo base» a «asistente»

**Objetivo:** ver cómo un modelo aprende a **responder preguntas** en lugar de solo continuar texto.

Un modelo base solo continúa el texto. Si le preguntas «¿Quién es Sancho?», puede que siga con
otra pregunta. Para crear asistentes como ChatGPT se hace un **ajuste supervisado (SFT)** con miles de
pares *instrucción → respuesta*. Aquí lo hacemos en miniatura con 30 pares sobre el Quijote y el Lazarillo
(archivo `data/qa/ejemplo_preguntas_respuestas.jsonl`).

La **pérdida solo se calcula sobre la respuesta**: el modelo no tiene que aprender a escribir preguntas.

> ⚠️ Con tan pocos ejemplos, el modelo aprende sobre todo el **formato** y el **tono**, no hechos fiables.
> Es un buen momento para hablar de **alucinaciones**.

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro)
import os, sys
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio e instalamos el paquete del taller
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    os.chdir(REPO_DIR)
    !pip install -q -e .
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
    if os.path.abspath("src") not in sys.path:
        sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## ⚙️ Ajustes

In [ ]:
from dhllm import load_config, set_seed
cfg = load_config(overrides={
    "sft": {
        # "base_model": "HuggingFaceTB/SmolLM2-135M-Instruct",  # más rápido, pero apenas sabe español
        # "epochs": 5,
    }
})
ajustes = cfg.sft
set_seed(cfg.seed)
PREGUNTAS = [
    "¿Cómo se llama el caballo de don Quijote?",
    "¿Quién fue el primer amo de Lázaro?",
    "¿Qué es la novela picaresca?",
]

## 1. Datos: pares pregunta-respuesta

In [ ]:
from dhllm.sft import load_qa
pares = load_qa(ajustes.qa_file)
pares[:3]

## 2. Modelo y respuestas ANTES

In [ ]:
from dhllm.finetune import load_base_model, prepare_model
from dhllm.sft import ask, format_prompt
modelo, tokenizer = load_base_model(ajustes.base_model)
print("Así ve el modelo una pregunta (plantilla de chat):\n")
print(format_prompt(tokenizer, PREGUNTAS[0]))
antes = {p: ask(modelo, tokenizer, p) for p in PREGUNTAS}
for p, r in antes.items():
    print(f"❓ {p}\n💬 {r}\n")

## 3. Entrenar

In [ ]:
from dhllm.sft import train_sft
modelo = prepare_model(modelo, ajustes.method, ajustes.lora_r, ajustes.lora_alpha)
historial = train_sft(modelo, tokenizer, pares, ajustes)

## 4. Respuestas DESPUÉS

In [ ]:
for p in PREGUNTAS:
    print(f"❓ {p}\n   ANTES:   {antes[p]}\n   DESPUÉS: {ask(modelo, tokenizer, p)}\n")

Prueba también una pregunta que **no** está en los datos:

In [ ]:
print(ask(modelo, tokenizer, "¿Quién escribió La Celestina?"))

## 🖐️ Tu turno
1. Escribe 10–20 pares pregunta-respuesta sobre **tu** corpus (mismo formato `.jsonl`) y entrena con ellos.
2. ¿Responde bien a preguntas nuevas? ¿Inventa? ¿Por qué?
3. **Para pensar:** en los grandes modelos, miles de personas escriben y valoran respuestas (RLHF/DPO).
   ¿Qué valores y qué visión del mundo entran así en el modelo?